# **FundusGuard AI**

model p2 using Vision Transformer

# **Collecting Dataset 1**

In [ ]:
!curl -L "https://ndownloader.figshare.com/files/7d73fc5157d44b221a62ed5a4baf3779" -o /content/PAPILA.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    62  100    62    0     0     76      0 --:--:-- --:--:-- --:--:--    76


In [ ]:
!unzip /content/PAPILA.zip -d /content/PAPILA

Archive:  /content/PAPILA.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/PAPILA.zip or
        /content/PAPILA.zip.zip, and cannot find /content/PAPILA.zip.ZIP, period.


In [ ]:
!file /content/PAPILA.zip

/content/PAPILA.zip: JSON data


**Figshare has it own WAP security for AWS s3 access, it is little hard to automate it with my colab.**

So i need to be MLOps mindset to this, I am mirror this dataset into my kaggle and then fetch it here.

In [ ]:
!kaggle datasets download -d labledata/papila-dataset-mirror-from-figshare \
    -p /content/

Dataset URL: https://www.kaggle.com/datasets/labledata/papila-dataset-mirror-from-figshare
License(s): GPL-3.0
100% 564M/564M [00:25<00:00, 23.3MB/s]



In [ ]:
!mkdir -p /content/PAPILA
!unzip -q /content/papila-dataset-mirror-from-figshare.zip \
    -d /content/PAPILA

Let's understand the architecture of this dataset

In [ ]:
import os

print("PAPILA dataset structure:\n")

for root, dirs, files in os.walk("/content/PAPILA"):
    level = root.replace("/content/PAPILA", "").count(os.sep)

    if level > 3:
        continue

    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:10]:
        print(f"{indent}    {file}")

PAPILA dataset structure:

PAPILA/
    PapilaDB-PAPILA-17f8fa7746adb20275b5b6a0d99dc9dfe3007e9f/
        README.md
        ExpertsSegmentations/
            ImagesWithContours/
                Opht_cont_RET240OD.jpg
                Opht_cont_RET224OS.jpg
                Opht_cont_RET087OS.jpg
                Opht_cont_RET023OS.jpg
                Opht_cont_RET174OD.jpg
                Opht_cont_RET246OS.jpg
                Opht_cont_RET006OS.jpg
                Opht_cont_RET048OD.jpg
                Opht_cont_RET186OD.jpg
                Opht_cont_RET051OD.jpg
            Contours/
                RET242OD_disc_exp2.txt
                RET039OD_disc_exp1.txt
                RET154OS_disc_exp1.txt
                RET210OD_cup_exp1.txt
                RET020OS_cup_exp2.txt
                RET122OS_disc_exp1.txt
                RET053OS_cup_exp1.txt
                RET089OD_disc_exp1.txt
                RET048OD_cup_exp1.txt
                RET010OD_cup_exp1.txt
        ClinicalData/
    

**Anatomy of PAPILA**

In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
papila_root = "/content/PAPILA"

# Find the long PapilaDB folder automatically for rename
folders = [
    f for f in os.listdir(papila_root)
    if os.path.isdir(os.path.join(papila_root, f))
]

print("Folders inside /content/PAPILA:")
for f in folders:
    print("  ", f)

if len(folders) != 1:
    raise Exception(
        "Expected exactly one dataset folder inside /content/PAPILA. "
        "Please check the folder structure."
    )

old_name = folders[0]
old_path = os.path.join(papila_root, old_name)
new_path = os.path.join(papila_root, "PapilaDB")

Folders inside /content/PAPILA:
   PapilaDB-PAPILA-17f8fa7746adb20275b5b6a0d99dc9dfe3007e9f


In [ ]:
if old_name != "PapilaDB":

    if os.path.exists(new_path):
        raise Exception(
            "A folder named 'PapilaDB' already exists. "
            "Please check before continuing."
        )

    os.rename(old_path, new_path)
    print("\nDataset folder renamed:")
    print("   ", old_name)
    print("   ↓")
    print("   PapilaDB")

else:
    print("\nPapilaDB already exists.")


Dataset folder renamed:
    PapilaDB-PAPILA-17f8fa7746adb20275b5b6a0d99dc9dfe3007e9f
   ↓
   PapilaDB


Reading the map controller excel files

In [ ]:
clinical_path = os.path.join(new_path, "ClinicalData")
fundus_path = os.path.join(new_path, "FundusImages")

od_file = os.path.join(clinical_path, "patient_data_od.xlsx")
os_file = os.path.join(clinical_path, "patient_data_os.xlsx")

# Check required files
required_files = [od_file, os_file, fundus_path]

for path in required_files:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file/folder not found: {path}")

print("\nDataset paths verified.")


Dataset paths verified.


In [ ]:
print("READING CLINICAL DATA")
print("-" * 30)

od = pd.read_excel(od_file)
os_data = pd.read_excel(os_file)

print("\nOD (Right Eye)")
print("Shape:", od.shape)

print("\nOS (Left Eye)")
print("Shape:", os_data.shape)

READING CLINICAL DATA
------------------------------

OD (Right Eye)
Shape: (246, 13)

OS (Left Eye)
Shape: (246, 13)


In [ ]:
print("COLOUMN INFORMATION")
print("-" * 30)

print("\nOD columns:")
for i, col in enumerate(od.columns):
    print(f"{i}: {col}")

print("\nOS columns:")
for i, col in enumerate(os_data.columns):
    print(f"{i}: {col}")

COLOUMN INFORMATION
------------------------------

OD columns:
0: Unnamed: 0
1: Age
2: Gender
3: Diagnosis
4: Refractive_Defect
5: Unnamed: 5
6: Unnamed: 6
7: Phakic/Pseudophakic
8: IOP
9: Unnamed: 9
10: Pachymetry
11: Axial_Length
12: VF_MD

OS columns:
0: Unnamed: 0
1: Age
2: Gender
3: Diagnosis
4: Refractive_Defect
5: Unnamed: 5
6: Unnamed: 6
7: Phakic/Pseudophakic
8: IOP
9: Unnamed: 9
10: Pachymetry
11: Axial_Length
12: VF_MD


In [ ]:
print("FIRST 5 ROWS")
print("-" * 30)

print("\nOD:")
display(od.head())

print("\nOS:")
display(os_data.head())

FIRST 5 ROWS
------------------------------

OD:


,Unnamed: 0,Age,Gender,Diagnosis,Refractive_Defect,Unnamed: 5,Unnamed: 6,Phakic/Pseudophakic,IOP,Unnamed: 9,Pachymetry,Axial_Length,VF_MD
0,NaN,Age,Gender,Diagnosis,dioptre_1,dioptre_2,astigmatism,Phakic/Pseudophakic,Pneumatic,Perkins,Pachymetry,Axial_Length,VF_MD
1,ID,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,#002,47,0,2,0.75,-1.75,90,0,21,NaN,586,23.64,-0.07
3,#004,58,1,1,1.5,-1.75,85,0,NaN,19,501,23.06,-3.26
4,#005,89,1,1,-0.75,-1.25,101,1,13,14,565,23.81,-14.98



OS:


,Unnamed: 0,Age,Gender,Diagnosis,Refractive_Defect,Unnamed: 5,Unnamed: 6,Phakic/Pseudophakic,IOP,Unnamed: 9,Pachymetry,Axial_Length,VF_MD
0,NaN,Age,Gender,Diagnosis,dioptre_1,dioptre_2,astigmatism,Phakic/Pseudophakic,Pneumatic,Perkins,Pachymetry,Axial_Length,VF_MD
1,ID,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,#002,47,0,2,-0.5,-1.5,88,0,20,NaN,603,23.77,0.17
3,#004,58,1,1,1.5,-2.5,85,1,NaN,19,511,22.96,-6.77
4,#005,89,1,1,-0.5,-2,100,1,24,10,575,24.33,-7.44


In [ ]:
def find_diagnosis_column(df):

    possible_names = [
        "diagnosis",
        "Diagnosis",
        "DIAGNOSIS",
        "diagnostic",
        "Diagnostic"
    ]

    for col in df.columns:
        if str(col).strip() in possible_names:
            return col

    # Fallback: search by keyword
    for col in df.columns:
        if "diagnos" in str(col).lower():
            return col

    return None

In [ ]:
od_diag_col = find_diagnosis_column(od)
os_diag_col = find_diagnosis_column(os_data)

print("DIAGNOSIS COLUMN")
print("-" * 30)

print("OD diagnosis column:", od_diag_col)
print("OS diagnosis column:", os_diag_col)

if od_diag_col is None or os_diag_col is None:
    print("\nDiagnosis column could not be automaticaly detected.")
    print("We will inspect the columns manually.")
else:
    print("DIAGNOSIS DISTRIBUTION")
    print("-" * 30)

    print("\nOD:")
    print(od[od_diag_col].value_counts(dropna=False).sort_index())

    print("\nOS:")
    print(os_data[os_diag_col].value_counts(dropna=False).sort_index())

    print("\nCombined:")
    combined_diag = pd.concat([
        od[od_diag_col],
        os_data[os_diag_col]
    ])

    print(combined_diag.value_counts(dropna=False).sort_index())

DIAGNOSIS COLUMN
------------------------------
OD diagnosis column: Diagnosis
OS diagnosis column: Diagnosis
DIAGNOSIS DISTRIBUTION
------------------------------

OD:


TypeError: '<' not supported between instances of 'str' and 'int'

In [ ]:
print("MISSING VALUE ANALYSIS")
print("-" * 30)

print("\nOD missing values:")
od_missing = od.isnull().sum()
print(od_missing[od_missing > 0])

print("\nOS missing values:")
os_missing = os_data.isnull().sum()
print(os_missing[os_missing > 0])

print("\nTotal missing cells:")
print("OD:", od.isnull().sum().sum())
print("OS:", os_data.isnull().sum().sum())

print("\nDUPLICATE CHECK")
print("-" * 30)


print("OD duplicate rows:", od.duplicated().sum())
print("OS duplicate rows:", os_data.duplicated().sum())

MISSING VALUE ANALYSIS
------------------------------

OD missing values:
Unnamed: 0               1
Age                      1
Gender                   1
Diagnosis                1
Refractive_Defect       15
Unnamed: 5               4
Unnamed: 6               4
Phakic/Pseudophakic      6
IOP                     48
Unnamed: 9             181
Pachymetry               8
Axial_Length             6
VF_MD                  163
dtype: int64

OS missing values:
Unnamed: 0               1
Age                      1
Gender                   1
Diagnosis                1
Refractive_Defect       13
Unnamed: 5               6
Unnamed: 6               6
Phakic/Pseudophakic      6
IOP                     46
Unnamed: 9             181
Pachymetry               8
Axial_Length             5
VF_MD                  163
dtype: int64

Total missing cells:
OD: 439
OS: 438

DUPLICATE CHECK
------------------------------
OD duplicate rows: 0
OS duplicate rows: 0


In [ ]:
print("FUNDUS IMAGE ANALYSIS")
print("-" * 30)

image_files = [
    f for f in os.listdir(fundus_path)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"))
]

print("Total fundus image files:", len(image_files))

print("\nFirst 20 images:")
for f in image_files[:20]:
    print(" ", f)

FUNDUS IMAGE ANALYSIS
------------------------------
Total fundus image files: 488

First 20 images:
  RET177OS.jpg
  RET221OD.jpg
  RET187OS.jpg
  RET293OD.jpg
  RET289OS.jpg
  RET237OS.jpg
  RET137OS.jpg
  RET024OD.jpg
  RET276OD.jpg
  RET096OD.jpg
  RET288OD.jpg
  RET023OD.jpg
  RET265OD.jpg
  RET213OD.jpg
  RET259OS.jpg
  RET234OS.jpg
  RET102OS.jpg
  RET136OS.jpg
  RET007OS.jpg
  RET170OD.jpg


So, we need uniquely identifies each image, better idea is create id for each image samples

In [ ]:

image_ids = {
    os.path.splitext(f)[0]: f
    for f in image_files
}


def possible_id_columns(df):
    result = []

    for col in df.columns:
        name = str(col).lower()

        if any(keyword in name for keyword in [
            "id",
            "patient",
            "image",
            "ret"
        ]):
            result.append(col)

    return result


print("POSSIBLE ID COLUMNS")
print("-" * 30)

print("OD:", possible_id_columns(od))
print("OS:", possible_id_columns(os_data))


for name, df in [("OD", od), ("OS", os_data)]:

    print("\n")
    print(name, "ID COLUMN VALUES")
    print("_" * 30)

    for col in possible_id_columns(df):

        print(f"\nColumn: {col}")
        print(df[col].head(15).tolist())


def find_image_id_column(df, image_ids):
    image_id_set = set(image_ids.keys())

    best_column = None
    best_matches = 0

    for col in df.columns:

        values = df[col].astype(str).str.strip()

        # Direct match
        matches = values.isin(image_id_set).sum()

        if matches > best_matches:
            best_matches = matches
            best_column = col

    return best_column, best_matches


od_id_col, od_matches = find_image_id_column(od, image_ids)
os_id_col, os_matches = find_image_id_column(os_data, image_ids)

print("IMAGE-ID MATCHING")
print("-" * 30)

print("\nOD:")
print("Possible image ID column:", od_id_col)
print("Direct image matches:", od_matches)

print("\nOS:")
print("Possible image ID column:", os_id_col)
print("Direct image matches:", os_matches)

POSSIBLE ID COLUMNS
------------------------------
OD: []
OS: []


OD ID COLUMN VALUES
______________________________


OS ID COLUMN VALUES
______________________________
IMAGE-ID MATCHING
------------------------------

OD:
Possible image ID column: None
Direct image matches: 0

OS:
Possible image ID column: None
Direct image matches: 0


In [ ]:
def normalize_id(value):

    if pd.isna(value):
        return None

    value = str(value).strip()

    # Remove extension if present
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]:
        if value.lower().endswith(ext):
            value = value[:-len(ext)]

    return value


def analyze_image_matching(df, id_col, image_ids, diagnosis_col, eye_name):
    if id_col is None:
        print(f"\nCannot perform direct matching for {eye_name}.")
        return None

    records = []

    for _, row in df.iterrows():
        image_id = normalize_id(row[id_col])

        diagnosis = (
            row[diagnosis_col]
            if diagnosis_col is not None
            else np.nan
        )

        image_exists = image_id in image_ids if image_id else False

        records.append({
            "image_id": image_id,
            "diagnosis": diagnosis,
            "image_exists": image_exists
        })

    result = pd.DataFrame(records)

    print(f"\n{eye_name} matching:")
    print("Excel records:", len(result))
    print("Records with image:", result["image_exists"].sum())
    print("Records without image:", (~result["image_exists"]).sum())

    return result


od_match = analyze_image_matching(
    od,
    od_id_col,
    image_ids,
    od_diag_col,
    "OD"
)

os_match = analyze_image_matching(
    os_data,
    os_id_col,
    image_ids,
    os_diag_col,
    "OS"
)


print("IMAGES WITHOUT EXCEL RECORDS")
print("-" * 30)

matched_ids = set()

if od_match is not None:
    matched_ids.update(
        od_match.loc[
            od_match["image_exists"],
            "image_id"
        ].dropna()
    )

if os_match is not None:
    matched_ids.update(
        os_match.loc[
            os_match["image_exists"],
            "image_id"
        ].dropna()
    )

unmatched_images = [
    filename
    for image_id, filename in image_ids.items()
    if image_id not in matched_ids
]

print("Total images:", len(image_files))
print("Images matched to Excel:", len(matched_ids))
print("Images without Excel match:", len(unmatched_images))

if unmatched_images:
    print("\nFirst 30 unmatched images:")
    for f in unmatched_images[:30]:
        print(" ", f)

print("\nRECORDS WITHOUT VALID DIAGNOSIS")
print("-" * 30)

if od_match is not None:

    missing_od = od_match[
        od_match["diagnosis"].isna()
    ]

    print("\nOD records without diagnosis:", len(missing_od))

if os_match is not None:

    missing_os = os_match[
        os_match["diagnosis"].isna()
    ]

    print("OS records without diagnosis:", len(missing_os))


print("\nUSABLE IMAGE RECORDS")
print("-" * 30)

usable_od = 0
usable_os = 0

if od_match is not None:

    usable_od_df = od_match[
        od_match["image_exists"] &
        od_match["diagnosis"].notna()
    ]

    usable_od = len(usable_od_df)

    print("OD usable images:", usable_od)

if os_match is not None:

    usable_os_df = os_match[
        os_match["image_exists"] &
        os_match["diagnosis"].notna()
    ]

    usable_os = len(usable_os_df)

    print("OS usable images:", usable_os)

print("\nTotal usable images:", usable_od + usable_os)


Cannot perform direct matching for OD.

Cannot perform direct matching for OS.
IMAGES WITHOUT EXCEL RECORDS
------------------------------
Total images: 488
Images matched to Excel: 0
Images without Excel match: 488

First 30 unmatched images:
  RET177OS.jpg
  RET221OD.jpg
  RET187OS.jpg
  RET293OD.jpg
  RET289OS.jpg
  RET237OS.jpg
  RET137OS.jpg
  RET024OD.jpg
  RET276OD.jpg
  RET096OD.jpg
  RET288OD.jpg
  RET023OD.jpg
  RET265OD.jpg
  RET213OD.jpg
  RET259OS.jpg
  RET234OS.jpg
  RET102OS.jpg
  RET136OS.jpg
  RET007OS.jpg
  RET170OD.jpg
  RET072OD.jpg
  RET284OD.jpg
  RET093OS.jpg
  RET056OD.jpg
  RET037OD.jpg
  RET189OS.jpg
  RET112OD.jpg
  RET239OS.jpg
  RET154OS.jpg
  RET135OD.jpg

RECORDS WITHOUT VALID DIAGNOSIS
------------------------------

USABLE IMAGE RECORDS
------------------------------

Total usable images: 0


Final summary of this analysis

In [ ]:
print("\nPAPILA DATASET AUDIT SUMMARY")
print("-" * 30)

print(f"""
Dataset folder : {new_path}

OD Excel records : {len(od)}
OS Excel records : {len(os_data)}

Fundus images : {len(image_files)}

OD usable records : {usable_od}
OS usable records : {usable_os}
Total usable records : {usable_od + usable_os}

OD missing cells : {od.isnull().sum().sum()}
OS missing cells : {os_data.isnull().sum().sum()}

OD duplicate rows : {od.duplicated().sum()}
OS duplicate rows : {os_data.duplicated().sum()}

Images without match : {len(unmatched_images)}
""")

print("Dataset audit completed.")
print("\nIMPORTANT: No images or Excel files was modified.")


PAPILA DATASET AUDIT SUMMARY
------------------------------

Dataset folder : /content/PAPILA/PapilaDB

OD Excel records : 246
OS Excel records : 246

Fundus images : 488

OD usable records : 0
OS usable records : 0
Total usable records : 0

OD missing cells : 439
OS missing cells : 438

OD duplicate rows : 0
OS duplicate rows : 0

Images without match : 488

Dataset audit completed.

IMPORTANT: No images or Excel files was modified.
